# 11 — Final Prediction and Deployment


> **Notebook 11 of 11** — part of the *Heart Disease Detection using Explainable AI* project.
> Run the notebooks **in order**, from 01 to 11.

---

## 🎯 Goal of this notebook

Bring everything together:

1. Load every saved model and predict for a **brand-new patient**
2. Show all three probabilities at once, plus a plain-English explanation
3. Demonstrate **batch prediction** on a whole file of patients
4. Write the `metadata.json` the website needs
5. **Verify** every saved file actually works
6. Launch the website

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt, seaborn as sns

warnings.filterwarnings("ignore"); sns.set_style("whitegrid")
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)
DATA, MODELS = "../data", "../models"

prep = np.load(f"{DATA}/prepared.npz", allow_pickle=True)
FEATURES = list(prep["features"])
X_train, X_test = prep["X_train"], prep["X_test"]
y_train, y_test = prep["y_train"], prep["y_test"]
scaler = joblib.load(f"{MODELS}/scaler.pkl")

MODEL_FILES = {"Logistic Regression": "logistic_regression",
               "Random Forest": "random_forest", "SVM": "svm"}
models = {n: joblib.load(f"{MODELS}/{f}.pkl") for n, f in MODEL_FILES.items()}

def predict_from_real_numbers(model):
    '''
    Our models were trained on SCALED numbers, but a human explanation must talk
    about REAL numbers (age 55, BP 140). This wrapper takes real numbers, scales
    them exactly as training did, and returns the probability of heart disease.
    SHAP and LIME use it so their output is readable.
    '''
    def inner(raw):
        raw = np.asarray(raw, dtype=float)
        if raw.ndim == 1:
            raw = raw.reshape(1, -1)
        return model.predict_proba(scaler.transform(raw))[:, 1]
    return inner

print("Loaded 3 models, the scaler, and the prepared data.")
print("Features:", FEATURES)

## 1. Predict for a brand-new patient

This is what the website does every time somebody moves a slider. Notice we pass **real** numbers
— the wrapper handles scaling behind the scenes.

In [ ]:
def make_patient(age, gender, height_cm, weight_kg, ap_hi, ap_lo,
                 cholesterol, gluc, smoke, alco, active):
    '''Turn plain human inputs into the 11 features the models expect.'''
    bmi = round(weight_kg / (height_cm / 100) ** 2, 2)
    return np.array([[age, gender, bmi, ap_hi, ap_lo, ap_hi - ap_lo,
                      cholesterol, gluc, smoke, alco, active]], dtype=float)

# A 64-year-old man with high blood pressure and poor lab results
new_patient = make_patient(age=64, gender=1, height_cm=168, weight_kg=97,
                           ap_hi=170, ap_lo=100, cholesterol=3, gluc=3,
                           smoke=1, alco=1, active=0)

print("New patient's 11 features:")
for name, value in zip(FEATURES, new_patient[0]):
    print(f"  {name:16s}: {value:g}")

In [ ]:
print("\n" + "=" * 56)
print("  WHAT EACH MODEL SAYS")
print("=" * 56)

probabilities = {}
for name, model in models.items():
    p = predict_from_real_numbers(model)(new_patient)[0]
    probabilities[name] = p
    bar = "#" * int(p * 40)
    print(f"  {name:<22}: {p*100:5.1f}%  |{bar}")

average = np.mean(list(probabilities.values()))
spread = max(probabilities.values()) - min(probabilities.values())

print("-" * 56)
print(f"  {'AVERAGE OF ALL THREE':<22}: {average*100:5.1f}%")
print(f"  {'Spread between models':<22}: {spread*100:5.1f} points")
print("=" * 56)

if average < 0.35:
    band, advice = "LOW RISK", "Mostly healthy signals."
elif average < 0.65:
    band, advice = "MODERATE RISK", "Some warning signs. A check-up is advisable."
else:
    band, advice = "HIGH RISK", "Several strong warning signs. Consult a doctor."
print(f"\n  VERDICT: {band} - {advice}")
print(f"  The three models are within {spread*100:.1f} points of each other,")
print("  so they clearly agree on this case.")

## 2. A plain-English explanation

A percentage alone is not useful to a patient. Here we translate the numbers into sentences, the
same way the website's Prediction page does.

In [ ]:
def explain_in_plain_english(features_row, avg_probability):
    values = dict(zip(FEATURES, features_row))
    risks, good = [], []

    if values["ap_hi"] >= 140 or values["ap_lo"] >= 90:
        risks.append(f"blood pressure is high at {values['ap_hi']:.0f}/{values['ap_lo']:.0f}")
    else:
        good.append("blood pressure is in a healthy range")

    if values["age_years"] >= 55:      risks.append("the patient is in an older age group")
    elif values["age_years"] < 45:     good.append("the patient is relatively young")

    if values["cholesterol"] >= 2:     risks.append("cholesterol is above normal")
    else:                              good.append("cholesterol is normal")

    if values["gluc"] >= 2:            risks.append("blood sugar is above normal")
    if values["bmi"] >= 30:            risks.append(f"BMI of {values['bmi']:.1f} is in the obese range")
    elif values["bmi"] < 25:           good.append("BMI is healthy")
    if values["smoke"]:                risks.append("the patient smokes")
    if not values["active"]:           risks.append("the patient is not physically active")
    else:                              good.append("the patient exercises regularly")

    text = (f"A {values['age_years']:.0f}-year-old "
            f"{'man' if values['gender'] else 'woman'}. ")
    text += ("Risk factors: " + ", ".join(risks) + ". ") if risks else "No major risk factors. "
    text += ("In their favour: " + ", ".join(good) + ". ") if good else ""
    text += (f"Averaging the three models, the estimated chance of cardiovascular "
             f"disease is {avg_probability*100:.1f}%.")
    return text

print(explain_in_plain_english(new_patient[0], average))

## 3. Batch prediction — a whole ward at once

In a real clinic you would not type patients in one at a time. You would upload a file. This is
exactly what the website's **Batch CSV Prediction** page does.

In [ ]:
ward = pd.DataFrame({
    "name":   ["Patient A", "Patient B", "Patient C", "Patient D", "Patient E"],
    "age":    [32, 52, 64, 41, 58],
    "gender": [0, 1, 1, 0, 1],
    "height": [165, 172, 168, 160, 175],
    "weight": [58, 80, 97, 55, 88],
    "ap_hi":  [112, 130, 170, 110, 150],
    "ap_lo":  [72, 85, 100, 70, 95],
    "cholesterol": [1, 1, 3, 1, 2],
    "gluc":   [1, 1, 3, 1, 1],
    "smoke":  [0, 0, 1, 0, 1],
    "alco":   [0, 0, 1, 0, 0],
    "active": [1, 1, 0, 1, 0]})

X_ward = np.column_stack([
    ward.age, ward.gender, (ward.weight / (ward.height / 100) ** 2).round(2),
    ward.ap_hi, ward.ap_lo, ward.ap_hi - ward.ap_lo,
    ward.cholesterol, ward.gluc, ward.smoke, ward.alco, ward.active]).astype(float)

results = ward[["name", "age", "ap_hi", "ap_lo", "cholesterol"]].copy()
for name, model in models.items():
    results[name] = (predict_from_real_numbers(model)(X_ward) * 100).round(1)

results["Average %"] = results[list(models)].mean(axis=1).round(1)
results["Risk"] = pd.cut(results["Average %"], [-1, 35, 65, 101],
                         labels=["Low", "Moderate", "High"])
results = results.sort_values("Average %", ascending=False)

print("The whole ward, ranked by risk - highest first:\n")
print(results.to_string(index=False))
print("\nThis is how a clinic would use the model: call the top patients in first.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colours = {"Low": "#10B981", "Moderate": "#F59E0B", "High": "#EF4444"}
bars = ax.barh(results.name, results["Average %"],
               color=[colours[r] for r in results.Risk])
ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.axvline(50, ls="--", color="grey", label="50% decision line")
ax.set_xlabel("Average predicted risk (%)"); ax.set_xlim(0, 110)
ax.set_title("Ward triage — who should the doctor see first?")
ax.legend()
plt.tight_layout(); plt.show()

## 4. Write the metadata file the website needs

In [ ]:
df_clean = pd.read_csv(f"{DATA}/cleaned_data.csv")
df_raw_rows = 70000
comparison = json.load(open(f"{MODELS}/comparison.json"))

metadata = {
    "features": FEATURES,
    "display": {
        "age_years": "Age (years)", "gender": "Gender", "bmi": "BMI (body mass index)",
        "ap_hi": "Systolic BP (upper number)", "ap_lo": "Diastolic BP (lower number)",
        "pulse_pressure": "Pulse pressure", "cholesterol": "Cholesterol level",
        "gluc": "Glucose level", "smoke": "Smoker", "alco": "Drinks alcohol",
        "active": "Physically active"},
    "plain": {
        "age_years": "How old the patient is. Heart risk goes up as age goes up.",
        "gender": "0 means female, 1 means male.",
        "bmi": "Weight compared to height. Over 25 is overweight, over 30 is obese.",
        "ap_hi": "The top blood pressure number. Over 140 is high.",
        "ap_lo": "The bottom blood pressure number. Over 90 is high.",
        "pulse_pressure": "Top BP minus bottom BP. A wide gap can mean stiff arteries.",
        "cholesterol": "Fat level in blood. 1 normal, 2 above normal, 3 well above normal.",
        "gluc": "Blood sugar level. 1 normal, 2 above normal, 3 well above normal.",
        "smoke": "0 means does not smoke, 1 means smokes.",
        "alco": "0 means does not drink, 1 means drinks alcohol.",
        "active": "0 means not physically active, 1 means physically active."},
    "dataset": {
        "rows_raw": df_raw_rows, "rows_clean": int(len(df_clean)),
        "removed": int(df_raw_rows - len(df_clean)), "n_features": len(FEATURES),
        "train": int(len(X_train)), "test": int(len(X_test)),
        "positive_rate": float(df_clean.cardio.mean() * 100)},
    "agreement": comparison["agreement"],
    "feature_means": {f: float(X_train[:, i].mean()) for i, f in enumerate(FEATURES)},
}
json.dump(metadata, open(f"{MODELS}/metadata.json", "w"), indent=2)
print("Saved ../models/metadata.json")

## 5. ✅ Final verification — does everything actually work?

Never hand in a project without this step. We check that every file exists, reload one model from
disk, and confirm it still predicts correctly.

In [ ]:
required = [
    "logistic_regression.pkl", "random_forest.pkl", "svm.pkl", "scaler.pkl",
    "metadata.json", "metrics.json", "comparison.json",
    "shap_background.npy", "lime_background.npy",
    "shap_X_sample.npy", "shap_y_sample.npy",
    "shapvals__logistic_regression.npy", "shapvals__random_forest.npy", "shapvals__svm.npy",
    "shapbase__logistic_regression.npy", "shapbase__random_forest.npy", "shapbase__svm.npy"]

print("Checking every file the website needs:\n")
missing, total_kb = [], 0
for f in required:
    path = f"{MODELS}/{f}"
    if os.path.exists(path):
        kb = os.path.getsize(path) / 1024
        total_kb += kb
        print(f"   OK      {f:36s} {kb:9,.1f} KB")
    else:
        missing.append(f)
        print(f"   MISSING {f}")

print(f"\n   Total size: {total_kb/1024:.1f} MB")
print("\nRESULT:", "ALL FILES PRESENT" if not missing else f"MISSING {len(missing)} FILES")

In [ ]:
# Reload from disk and prove the saved models still work
print("Reloading each model from disk and testing on 5 real test patients...\n")

for name, short in MODEL_FILES.items():
    reloaded = joblib.load(f"{MODELS}/{short}.pkl")
    probs = reloaded.predict_proba(scaler.transform(X_test[:5]))[:, 1]
    print(f"   {name:<22}: {np.round(probs * 100, 1)} %")

print(f"\n   Actual answers        : "
      f"{['DISEASE' if v else 'healthy' for v in y_test[:5]]}")
print("\nEverything reloads and predicts correctly. The project is ready.")

---
## 6. 🚀 Launch the website

Open a terminal in the **project root folder** (one level up from `notebooks/`) and run:

```bash
streamlit run app.py
```

Your browser will open at `http://localhost:8501` with seven interactive pages:

| Page | What it shows |
|---|---|
| 🏠 Dashboard | KPI cards and interactive charts exploring the data |
| 🩺 Patient Prediction | All three models' probabilities at once, with gauges |
| 🧠 SHAP Explanation | Live SHAP for any patient and any model |
| 🍋 LIME Explanation | Live LIME rules for any patient and any model |
| 📁 Batch CSV Prediction | Upload a file, score every patient, download results |
| 📊 Model Performance | Metrics, confusion matrices, ROC curves, cross-validation |
| ℹ️ About the Project | The full story, plus viva questions and answers |

---

## 🎓 The complete project in one page

| Notebook | What it did | Key result |
|---|---|---|
| 01 | Understood the data | 70,000 patients, balanced, but **dirty** |
| 02 | Cleaned it | Removed 1,948 impossible rows (2.78%) |
| 03 | Explored it | BP, age and cholesterol matter; self-reported habits do not |
| 04 | Engineered and split | 11 features, 80/20 split, scaled without leakage |
| 05 | Logistic Regression | ~73% accuracy, fully transparent coefficients |
| 06 | Random Forest | ~73% accuracy, best ROC-AUC (~0.80) |
| 07 | SVM | ~73% accuracy, highest precision |
| 08 | Compared them | **Statistically indistinguishable**; agree within ~5 points |
| 09 | SHAP | Contributions add up exactly; found the 140 mmHg threshold |
| 10 | LIME | Independent rules that **agree with SHAP** |
| 11 | Deployment | Everything verified and running in a live website |

### Three things to say in your viva

1. **"The raw data contained blood pressure readings of 16,020 mmHg."** We found it, removed it
   with medically justified rules, and reported the exact count.

2. **"73% is the honest ceiling for this dataset."** Published research agrees. Anyone reporting
   95% has leaked test data into training.

3. **"The three models are statistically indistinguishable, so I would deploy the most
   interpretable one."** Performance is not the only criterion in medicine.

---

*This is an academic project for educational purposes. It is not a medical device, has not been
clinically validated, and must never be used for real health decisions.*